# Vision-Based Road Sign & Hazard Detection 
### ECE 284/484 Final Project
Christina Falasco, Ryan Gae, Felipe Villanueva, Lucy Yang

Ensure that YOLOv5 has been cloned to the workspace! This project should include the full YOLOv5 folder (clone below!)

Make sure that the config.yaml contains all necessary information and paths to relevant folders. See the following videos for guides:

##### YOLO Guides:
5: https://www.youtube.com/watch?v=MdF6x6ZmLAY

8: https://www.youtube.com/watch?v=m9fH9OWn8YM

In [ ]:
%git clone https://github.com/ultralytics/yolov5.git
%pip install -r yolov5/requirements.txt

  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached thop-0.1.1.post2209072238-py3-none-any.whl.metadata (2.7 kB)
  Using cached ultralytics-8.4.42-py3-none-any.whl.metadata (39 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached polars-1.40.1-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.19-py3-none-any.whl.metadata (14 kB)
  Using cached polars_runtime_32-1.40.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.5 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (72.9 MB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached thop-0.1.1.post2209072238-py3-none-any.whl (15 kB)
Using cached ultralytics-8.4.42-py3-none-any.whl

In [2]:
import cv2
import numpy as np
import torch
import yaml
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image
from utils.utils import plot_results


if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


#### Modified YOLOv5s derived from Qiu et. al.
https://www.nature.com/articles/s41598-024-62629-4

##### Make sure to have the following examples inside respective files!


yolov5/models/common.py:
```
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        scale = self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))
        return x * scale.unsqueeze(-1).unsqueeze(-1)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx, _ = x.max(dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, spatial_kernel=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_kernel)

    def forward(self, x):
        return self.sa(self.ca(x))


class CBAMC3(nn.Module):
    """C3 replacement: adds CBAM attention + GELU activations (paper Section 2.2)."""
    def __init__(self, c1, c2, n=1, shortcut=True, g=1, e=0.5):
        super().__init__()
        c_ = int(c2 * e)

        def _conv(ci, co, k):
            return nn.Sequential(
                nn.Conv2d(ci, co, k, padding=k // 2, bias=False),
                nn.BatchNorm2d(co),
                nn.GELU(),
            )

        self.cv1  = _conv(c1, c_, 1)
        self.cv2  = _conv(c1, c_, 1)
        self.cv3  = _conv(2 * c_, c2, 1)
        self.m    = nn.Sequential(*[Bottleneck(c_, c_, shortcut, g, e=1.0) for _ in range(n)])
        self.cbam = CBAM(c2)

    def forward(self, x):
        return self.cbam(self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), dim=1)))
```

yolov5/models/yolo.py:

add [CBAM, CBAMC3] to the import list inside yolo.py (from models.common import...)

add [CBAM, CBAMC3] to the list inside parse_model around line 407 (if m in {...)

add [CBAM, CBAMC3] to ~line 432 (if m in {BottleneckCSP, C3, C3TR, C3Ghost, C3x})

In [15]:
# modify YOLOv5s
num_classes = 0  # change this when we find out how many classes we have

with open("yolov5/models/qiu_yolov5s.yaml", "w") as f:
    f.write("nc: " + str(num_classes) + "\n")
    f.write("depth_multiple: 0.33" + "\n")
    f.write("width_multiple: 0.50" + "\n")
    f.write("\n")

    f.write("anchors:" + "\n")
    f.write("  - [10,13, 16,30, 33,23]" + "\n")
    f.write("  - [30,61, 62,45, 59,119]" + "\n")
    f.write("  - [116,90, 156,198, 373,326]" + "\n")
    f.write("\n")

    f.write("backbone:" + "\n")
    f.write("  - [-1, 1, Conv, [64, 6, 2, 2]]" + "\n")
    f.write("  - [-1, 1, Conv, [128, 3, 2]]" + "\n")
    f.write("  - [-1, 3, CBAMC3, [128]]" + "\n")
    f.write("  - [-1, 1, Conv, [256, 3, 2]]" + "\n")
    f.write("  - [-1, 6, CBAMC3, [256]]" + "\n")
    f.write("  - [-1, 1, Conv, [512, 3, 2]]" + "\n")
    f.write("  - [-1, 9, CBAMC3, [512]]" + "\n")
    f.write("  - [-1, 1, Conv, [1024, 3, 2]]" + "\n")
    f.write("  - [-1, 3, CBAMC3, [1024]]" + "\n")
    f.write("  - [-1, 1, SPPF, [1024, 5]]" + "\n")
    f.write("\n")

    f.write("head:" + "\n")
    f.write("  - [-1, 1, Conv, [512, 1, 1]]" + "\n")
    f.write("  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]" + "\n")
    f.write("  - [[-1,6], 1, Concat, [1]]" + "\n")
    f.write("  - [-1, 3, C3, [512, False]]" + "\n")
    f.write("  - [-1, 1, Conv, [256, 1, 1]]" + "\n")
    f.write("  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]" + "\n")
    f.write("  - [[-1,4], 1, Concat, [1]]" + "\n")
    f.write("  - [-1, 3, C3, [256, False]]" + "\n")
    f.write("  - [-1, 1, Conv, [256, 3, 2]]" + "\n")
    f.write("  - [[-1,14], 1, Concat, [1]]" + "\n")
    f.write("  - [-1, 3, C3, [512, False]]" + "\n")
    f.write("  - [-1, 1, Conv, [512, 3, 2]]" + "\n")
    f.write("  - [[-1,10], 1, Concat, [1]]" + "\n")
    f.write("  - [-1, 3, C3, [1024, False]]" + "\n")
    f.write("  - [[17,20,23], 1, Detect, [nc, anchors]]" + "\n")

In [16]:
# change parameters accordingly
!python yolov5/train.py \
    --img 640 \
    --batch 16 \
    --epochs 50 \
    --data 'config.yaml' \
    --cfg yolov5/models/qiu_yolov5s.yaml \
    --weights '' \
    --project /home/sagemaker-user/finalproj/qiu_yolov5s_results \
    --cache 

2026-04-28 00:53:28.819689: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777337608.836734    1164 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777337608.841987    1164 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
train: weights=, cfg=yolov5/models/qiu_yolov5s.yaml, data=config.yaml, hyp=yolov5/data/hyps/hyp.scratch-low.yaml, epochs=50, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=yolov5/data/hyps, resume_evolve=None, bucket=, cache=ram, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=8, p

In [ ]:
plot_results()
Image(filename='results.png', width=1000)

In [ ]:
# using small model
model = YOLO("yolov8s.yaml")  # consider using nano for size/computing constraints ?

res8 = model.train(data="config.yaml",
                  epochs=50,
                  degrees=5.0,
                  hsv_h=0.015,
                  hsv_v=0.4,
                  project="/home/sagemaker-user/finalproj/yolov8_results")